# E6 - Preprocesamiento axial Al-Kafri/Sudirman: pairing imagen-ground truth

Este notebook usa los inventarios del notebook 12 para construir un primer emparejamiento trazable entre imagenes axiales `.ima` y archivos reales de ground truth. No entrena modelos y no fuerza emparejamientos ambiguos.

Objetivo: dejar candidatos, evidencia visual, sanity checks y, solo si corresponde, un `pairing_v1` procesado para que el notebook 14 pueda entrenar un primer baseline axial.

In [12]:
# Dependencias para Google Colab.
try:
    import pydicom  # noqa: F401
    import skimage  # noqa: F401
except Exception:
    %pip install -q pydicom scikit-image

In [13]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy.io import loadmat, whosmat
from skimage.transform import resize
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 180)

## Montaje de Drive y rutas

In [14]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
ALKAFRI_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI")
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
NESTED_EXTRACTED_ROOT = EXTRACTED_ROOT / "_nested"
INVENTORY_RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory")
PAIRING_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing")
PROCESSED_ROOT = ALKAFRI_ROOT / "processed"
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

PAIRING_V1_ROOT = PROCESSED_ROOT / "pairing_v1"

for path in [PAIRING_ROOT, PROCESSED_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "extracted_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_extracted_file_inventory.csv",
    "series_orientation": INVENTORY_RESULTS_ROOT / "E6_alkafri_series_orientation_candidates.csv",
    "ground_truth_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_ground_truth_inventory.csv",
    "dicom_metadata_sample": INVENTORY_RESULTS_ROOT / "E6_alkafri_dicom_metadata_sample.csv",
    "dicom_reading_report": INVENTORY_RESULTS_ROOT / "E6_alkafri_dicom_reading_report.csv",
    "inventory_report": INVENTORY_RESULTS_ROOT / "E6_alkafri_inventory_report.json",
}

missing_inputs = {name: str(path) for name, path in INPUTS.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(missing_inputs)

print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("PAIRING_ROOT:", PAIRING_ROOT)
print("PAIRING_V1_ROOT:", PAIRING_V1_ROOT)

ALKAFRI_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI
PAIRING_ROOT: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing
PAIRING_V1_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/processed/pairing_v1


## Carga de inventarios del notebook 12

In [16]:
extracted_inventory_df = pd.read_csv(INPUTS["extracted_inventory"])
series_orientation_df = pd.read_csv(INPUTS["series_orientation"])
ground_truth_inventory_df = pd.read_csv(INPUTS["ground_truth_inventory"])
dicom_metadata_sample_df = pd.read_csv(INPUTS["dicom_metadata_sample"])
dicom_reading_report_df = pd.read_csv(INPUTS["dicom_reading_report"])
with open(INPUTS["inventory_report"], "r", encoding="utf-8") as f:
    inventory_report = json.load(f)

for df in [extracted_inventory_df, series_orientation_df, ground_truth_inventory_df]:
    if "extension" in df.columns:
        df["extension"] = df["extension"].astype(str).str.lower()

summary = {
    "total_ima_inventory": int((extracted_inventory_df.get("extension", pd.Series(dtype=str)) == ".ima").sum()),
    "valid_dicom_orientation_rows": int(series_orientation_df.get("valid_medical_dicom", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if "valid_medical_dicom" in series_orientation_df.columns else int(len(series_orientation_df)),
    "axial_candidates": int((series_orientation_df.get("orientation_candidate", pd.Series(dtype=str)) == "axial_candidate").sum()) if "orientation_candidate" in series_orientation_df.columns else 0,
    "sagittal_candidates": int((series_orientation_df.get("orientation_candidate", pd.Series(dtype=str)) == "sagittal_candidate").sum()) if "orientation_candidate" in series_orientation_df.columns else 0,
    "ground_truth_files": int(len(ground_truth_inventory_df)),
    "png_inventory": int((extracted_inventory_df.get("extension", pd.Series(dtype=str)) == ".png").sum()),
    "mat_inventory": int((extracted_inventory_df.get("extension", pd.Series(dtype=str)) == ".mat").sum()),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

/tmp/ipykernel_8201/1745849731.py:3: DtypeWarning: Columns (27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  ground_truth_inventory_df = pd.read_csv(INPUTS["ground_truth_inventory"])


{
  "total_ima_inventory": 17497,
  "valid_dicom_orientation_rows": 17496,
  "axial_candidates": 8203,
  "sagittal_candidates": 7646,
  "ground_truth_files": 37129,
  "png_inventory": 29361,
  "mat_inventory": 32
}


## Filtrado de imagenes axiales utiles

In [17]:
def text_has_any(value, keywords):
    text = str(value or "").lower()
    return any(k.lower() in text for k in keywords)


def combined_text(row):
    parts = []
    for col in ["relative_path", "file_path", "SeriesDescription", "ProtocolName", "parent_folder"]:
        if col in row and pd.notna(row[col]):
            parts.append(str(row[col]))
    return " ".join(parts).lower()


if "valid_medical_dicom" not in series_orientation_df.columns:
    series_orientation_df["valid_medical_dicom"] = True
if "extension" not in series_orientation_df.columns:
    series_orientation_df["extension"] = ""

axial_ima_df = series_orientation_df[
    series_orientation_df["valid_medical_dicom"].fillna(False).astype(bool)
    & series_orientation_df["orientation_candidate"].eq("axial_candidate")
    & series_orientation_df["extension"].astype(str).str.lower().eq(".ima")
].copy()

if len(axial_ima_df) > 0:
    axial_ima_df["combined_text"] = axial_ima_df.apply(combined_text, axis=1)
    axial_ima_df["is_localizer"] = axial_ima_df["combined_text"].str.contains("localizer|localiser|survey", case=False, regex=True, na=False)
    axial_ima_df["priority_text_hit"] = axial_ima_df["combined_text"].str.contains("t2_tse_tra|t1_tse_tra|posdisp|axial|\btra\b|\bax\b", case=False, regex=True, na=False)
    axial_ima_df["sequence_candidate"] = np.select(
        [
            axial_ima_df["combined_text"].str.contains("t2", case=False, na=False),
            axial_ima_df["combined_text"].str.contains("t1", case=False, na=False),
        ],
        ["T2", "T1"],
        default="unknown",
    )
    axial_ima_df = axial_ima_df[~axial_ima_df["is_localizer"]].copy()
    axial_ima_df = axial_ima_df.sort_values(["priority_text_hit", "SeriesInstanceUID", "InstanceNumber"], ascending=[False, True, True])
else:
    axial_ima_df["combined_text"] = []
    axial_ima_df["is_localizer"] = []
    axial_ima_df["priority_text_hit"] = []
    axial_ima_df["sequence_candidate"] = []

axial_ima_candidates_csv_path = PAIRING_ROOT / "E6_alkafri_axial_ima_candidates.csv"
axial_ima_df.to_csv(axial_ima_candidates_csv_path, index=False)

print("Axial IMA candidates:", len(axial_ima_df))
print("CSV:", axial_ima_candidates_csv_path)
display(axial_ima_df.head())

Axial IMA candidates: 8150
CSV: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_axial_ima_candidates.csv


,file_path,relative_path,extension,read_ok,valid_medical_dicom,read_error,discard_reason,PatientID,StudyInstanceUID,SeriesInstanceUID,SeriesDescription,ProtocolName,Modality,ImageOrientationPatient,ImagePositionPatient,PixelSpacing,SliceThickness,Rows,Columns,InstanceNumber,orientation_candidate,classification_reason,combined_text,is_localizer,priority_text_hit,sequence_candidate
3249,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,.ima,True,True,NaN,NaN,NaN,1.3.12.2.1107.5.2.40.50233.3000001509190623393...,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,t2_tse_tra_384,NaN,MR,"[""0.9997236025665"", ""0.0123649674503"", ""0.0199...","[""-98.338610780129"", ""-80.155962773177"", ""51.6...","[""0.6875"", ""0.6875""]",4.0,320,320,1,axial_candidate,path_keyword,_nested/main_dataset__mri_data/01_mri_data/003...,False,True,T2
3250,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,.ima,True,True,NaN,NaN,NaN,1.3.12.2.1107.5.2.40.50233.3000001509190623393...,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,t2_tse_tra_384,NaN,MR,"[""0.9997236025665"", ""0.0123649674503"", ""0.0199...","[""-98.244200837502"", ""-80.738034077498"", ""47.2...","[""0.6875"", ""0.6875""]",4.0,320,320,2,axial_candidate,path_keyword,_nested/main_dataset__mri_data/01_mri_data/003...,False,True,T2
3251,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,.ima,True,True,NaN,NaN,NaN,1.3.12.2.1107.5.2.40.50233.3000001509190623393...,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,t2_tse_tra_384,NaN,MR,"[""0.9997236025665"", ""0.0123649674503"", ""0.0199...","[""-98.149789941201"", ""-81.32010538182"", ""42.92...","[""0.6875"", ""0.6875""]",4.0,320,320,3,axial_candidate,path_keyword,_nested/main_dataset__mri_data/01_mri_data/003...,False,True,T2
3252,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,.ima,True,True,NaN,NaN,NaN,1.3.12.2.1107.5.2.40.50233.3000001509190623393...,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,t2_tse_tra_384,NaN,MR,"[""0.9997236025665"", ""0.0123649674503"", ""0.0199...","[""-98.055379044899"", ""-81.902176686141"", ""38.5...","[""0.6875"", ""0.6875""]",4.0,320,320,4,axial_candidate,path_keyword,_nested/main_dataset__mri_data/01_mri_data/003...,False,True,T2
3253,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,.ima,True,True,NaN,NaN,NaN,1.3.12.2.1107.5.2.40.50233.3000001509190623393...,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,t2_tse_tra_384,NaN,MR,"[""0.9997499900683"", ""0.0009263999362"", ""0.0223...","[""-98.164838175663"", ""-78.835292605798"", ""11.4...","[""0.6875"", ""0.6875""]",4.0,320,320,5,axial_candidate,path_keyword,_nested/main_dataset__mri_data/01_mri_data/003...,False,True,T2


## Agrupacion por paciente/serie

In [18]:
def parse_number_list(values):
    parsed = []
    for value in values:
        try:
            if pd.notna(value):
                parsed.append(int(float(value)))
        except Exception:
            pass
    return sorted(set(parsed))


def common_path(paths):
    paths = [str(p) for p in paths if pd.notna(p)]
    if not paths:
        return ""
    return os.path.commonpath(paths)


series_rows = []
if len(axial_ima_df) > 0:
    for keys, group in axial_ima_df.groupby(["PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SeriesDescription"], dropna=False):
        patient_id, study_uid, series_uid, series_desc = keys
        instance_numbers = parse_number_list(group.get("InstanceNumber", []))
        row = {
            "PatientID": patient_id,
            "StudyInstanceUID": study_uid,
            "SeriesInstanceUID": series_uid,
            "SeriesDescription": series_desc,
            "n_slices": int(len(group)),
            "Rows": group["Rows"].dropna().astype(str).mode().iloc[0] if "Rows" in group and group["Rows"].notna().any() else None,
            "Columns": group["Columns"].dropna().astype(str).mode().iloc[0] if "Columns" in group and group["Columns"].notna().any() else None,
            "PixelSpacing": group["PixelSpacing"].dropna().astype(str).mode().iloc[0] if "PixelSpacing" in group and group["PixelSpacing"].notna().any() else None,
            "SliceThickness": group["SliceThickness"].dropna().astype(str).mode().iloc[0] if "SliceThickness" in group and group["SliceThickness"].notna().any() else None,
            "InstanceNumber_min": min(instance_numbers) if instance_numbers else None,
            "InstanceNumber_max": max(instance_numbers) if instance_numbers else None,
            "InstanceNumber_unique": json.dumps(instance_numbers),
            "files_ordered": json.dumps(group.sort_values("InstanceNumber", na_position="last")["file_path"].tolist(), ensure_ascii=False),
            "relative_paths_ordered": json.dumps(group.sort_values("InstanceNumber", na_position="last")["relative_path"].tolist(), ensure_ascii=False),
            "common_path": common_path(group["file_path"].tolist()),
            "sequence_candidate": group["sequence_candidate"].dropna().astype(str).mode().iloc[0] if "sequence_candidate" in group and len(group) else "unknown",
            "possible_level": None,
        }
        level_match = re.search(r"(l[1-5]|s1)", " ".join(group["relative_path"].astype(str)).lower())
        if level_match:
            row["possible_level"] = level_match.group(1).upper()
        series_rows.append(row)

axial_series_summary_df = pd.DataFrame(series_rows)
axial_series_summary_csv_path = PAIRING_ROOT / "E6_alkafri_axial_series_summary.csv"
axial_series_summary_df.to_csv(axial_series_summary_csv_path, index=False)

print("Series axiales:", len(axial_series_summary_df))
display(axial_series_summary_df.head())

Series axiales: 655


,PatientID,StudyInstanceUID,SeriesInstanceUID,SeriesDescription,n_slices,Rows,Columns,PixelSpacing,SliceThickness,InstanceNumber_min,InstanceNumber_max,InstanceNumber_unique,files_ordered,relative_paths_ordered,common_path,sequence_candidate,possible_level
0,NaN,1.2.840.113845.11.1000000002066606790.20151215...,1.3.12.2.1107.5.2.40.50233.2015121411391613634...,t2_tse_tra_384,12,320,320,"[""0.6875"", ""0.6875""]",4.0,1,12,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]","[""/content/drive/MyDrive/PFI_MVP/data/AXIAL_AL...","[""_nested/main_dataset__MRI_Data/01_MRI_Data/0...",/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,T2,None
1,NaN,1.2.840.113845.11.1000000002066606790.20151215...,1.3.12.2.1107.5.2.40.50233.2015121411412070258...,t1_tse_tra,12,320,320,"[""0.6875"", ""0.6875""]",4.0,1,12,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]","[""/content/drive/MyDrive/PFI_MVP/data/AXIAL_AL...","[""_nested/main_dataset__MRI_Data/01_MRI_Data/0...",/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,T1,None
2,NaN,1.2.840.113845.11.1000000002066606790.20151215...,1.3.12.2.1107.5.2.40.50233.3000001512141040083...,PosDisp: [4] t2_tse_tra_384,3,512,512,"[""0.68359375"", ""0.68359375""]",8.0,5,7,"[5, 6, 7]","[""/content/drive/MyDrive/PFI_MVP/data/AXIAL_AL...","[""_nested/main_dataset__MRI_Data/01_MRI_Data/0...",/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,T2,None
3,NaN,1.2.840.113845.11.1000000002066606790.20151218...,1.3.12.2.1107.5.2.40.50233.2015121709403875748...,t2_tse_tra_384,20,320,320,"[""0.6875"", ""0.6875""]",4.0,1,20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[""/content/drive/MyDrive/PFI_MVP/data/AXIAL_AL...","[""_nested/main_dataset__MRI_Data/01_MRI_Data/0...",/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,T2,None
4,NaN,1.2.840.113845.11.1000000002066606790.20151218...,1.3.12.2.1107.5.2.40.50233.2015121709444396204...,t1_tse_tra,20,320,320,"[""0.6875"", ""0.6875""]",4.0,1,20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[""/content/drive/MyDrive/PFI_MVP/data/AXIAL_AL...","[""_nested/main_dataset__MRI_Data/01_MRI_Data/0...",/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,T1,None


## Ground truth real desde ZIPs internos

In [19]:
def is_real_gt_row(row):
    rel = str(row.get("relative_path", "")).lower()
    path = str(row.get("file_path", "")).lower()
    ext = str(row.get("extension", "")).lower()
    name = str(row.get("file_name", "")).lower()
    in_nested_gt = (
        "_nested" in path
        and ("ground_truth__manual_label_data" in path or "ground_truth__ground_truth_label" in path)
    )
    relevant_ext = ext in [".png", ".mat", ".xcf", ".csv", ".jpg", ".jpeg", ".bmp"]
    not_source = "source_code" not in path
    not_aux_figure = not name.startswith("figure")
    return in_nested_gt and relevant_ext and not_source and not_aux_figure


gt_real_df = extracted_inventory_df[extracted_inventory_df.apply(is_real_gt_row, axis=1)].copy()
gt_real_csv_path = PAIRING_ROOT / "E6_alkafri_ground_truth_real_files.csv"
gt_real_df.to_csv(gt_real_csv_path, index=False)

print("Ground truth real files:", len(gt_real_df))
display(gt_real_df.head())

Ground truth real files: 37080


,file_path,relative_path,file_name,extension,parent_folder,size_bytes,probable_source,probable_type,is_nested,nested_source_zip,has_axial,has_sagittal,has_sag,has_mask,has_label,has_ground,has_gt,has_segmentation,has_disc,has_posterior,has_thecal,has_canal,has_intervertebral,has_l3,has_l4,has_l5,has_s1
17616,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D3.png,.png,Labeller 01,1019,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False
17617,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D3.xcf,.xcf,Labeller 01,105656,ground_truth,unknown,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False
17618,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D4.png,.png,Labeller 01,1115,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False
17619,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D4.xcf,.xcf,Labeller 01,105618,ground_truth,unknown,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False
17620,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D5.png,.png,Labeller 01,1084,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False


## Exploracion de estructura de ground truth

In [20]:
TOKEN_REGEX = re.compile(r"\d+|l[1-5]|s1", flags=re.IGNORECASE)


def numeric_tokens_from_text(text):
    return TOKEN_REGEX.findall(str(text))


def infer_patient_token(path_text):
    tokens = numeric_tokens_from_text(path_text)
    return tokens[0] if tokens else None


def infer_gt_type(path_text):
    text = str(path_text).lower()
    if "manual" in text:
        return "manual_label"
    if "ground_truth" in text or "ground truth" in text:
        return "final_ground_truth"
    if "intermedi" in text:
        return "intermediary_ground_truth"
    return "unknown"


def infer_structure_label(path_text):
    text = str(path_text).lower()
    hits = [k for k in ["disc", "posterior", "thecal", "canal", "intervertebral", "mask", "label"] if k in text]
    return "|".join(hits) if hits else "unknown"


gt_token_rows = []
for _, row in gt_real_df.iterrows():
    text = f"{row.get('relative_path', '')} {row.get('parent_folder', '')} {row.get('file_name', '')}"
    gt_token_rows.append({
        "file_path": row.get("file_path"),
        "relative_path": row.get("relative_path"),
        "file_name": row.get("file_name"),
        "extension": row.get("extension"),
        "patient_id_candidate": infer_patient_token(text),
        "numeric_tokens": json.dumps(numeric_tokens_from_text(text)),
        "gt_type": infer_gt_type(text),
        "structure_candidate": infer_structure_label(text),
        "possible_level": "|".join([t.upper() for t in numeric_tokens_from_text(text) if re.match(r"l[1-5]|s1", t, re.I)]),
    })

gt_path_tokens_df = pd.DataFrame(gt_token_rows)
gt_path_tokens_csv_path = PAIRING_ROOT / "E6_alkafri_ground_truth_path_tokens.csv"
gt_path_tokens_df.to_csv(gt_path_tokens_csv_path, index=False)

gt_folder_summary_df = (
    gt_path_tokens_df.groupby(["gt_type", "structure_candidate", "extension"], dropna=False)
    .agg(n_files=("file_path", "count"))
    .reset_index()
    if len(gt_path_tokens_df) else pd.DataFrame(columns=["gt_type", "structure_candidate", "extension", "n_files"])
)
gt_folder_summary_csv_path = PAIRING_ROOT / "E6_alkafri_ground_truth_folder_summary.csv"
gt_folder_summary_df.to_csv(gt_folder_summary_csv_path, index=False)

display(gt_path_tokens_df.head())
display(gt_folder_summary_df.head())

,file_path,relative_path,file_name,extension,patient_id_candidate,numeric_tokens,gt_type,structure_candidate,possible_level
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D3.png,.png,03,"[""03"", ""01"", ""1"", ""0001"", ""3"", ""01"", ""1"", ""000...",manual_label,label,
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D3.xcf,.xcf,03,"[""03"", ""01"", ""1"", ""0001"", ""3"", ""01"", ""1"", ""000...",manual_label,label,
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D4.png,.png,03,"[""03"", ""01"", ""1"", ""0001"", ""4"", ""01"", ""1"", ""000...",manual_label,label,
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D4.xcf,.xcf,03,"[""03"", ""01"", ""1"", ""0001"", ""4"", ""01"", ""1"", ""000...",manual_label,label,
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D5.png,.png,03,"[""03"", ""01"", ""1"", ""0001"", ""5"", ""01"", ""1"", ""000...",manual_label,label,


,gt_type,structure_candidate,extension,n_files
0,final_ground_truth,label,.png,12360
1,final_ground_truth,mask|label,.png,1545
2,manual_label,label,.png,15450
3,manual_label,label,.xcf,7725


## Lectura de archivos MAT

In [21]:
mat_rows = []
mat_files = gt_real_df[gt_real_df["extension"].eq(".mat")].copy() if len(gt_real_df) else pd.DataFrame()
for _, row in tqdm(mat_files.iterrows(), total=len(mat_files), desc="Leyendo MAT"):
    path = row["file_path"]
    try:
        data = loadmat(path)
        for key, value in data.items():
            if key in {"__header__", "__version__", "__globals__"}:
                continue
            arr = np.asarray(value)
            mat_rows.append({
                "file_path": path,
                "relative_path": row.get("relative_path"),
                "variable": key,
                "shape": str(tuple(arr.shape)),
                "dtype": str(arr.dtype),
                "ndim": int(arr.ndim),
                "numeric": bool(np.issubdtype(arr.dtype, np.number)),
                "mask_candidate": bool(np.issubdtype(arr.dtype, np.number) and arr.ndim in [2, 3]),
                "error": None,
            })
    except Exception as exc:
        mat_rows.append({
            "file_path": path,
            "relative_path": row.get("relative_path"),
            "variable": None,
            "shape": None,
            "dtype": None,
            "ndim": None,
            "numeric": False,
            "mask_candidate": False,
            "error": repr(exc),
        })

mat_variables_summary_df = pd.DataFrame(mat_rows)
mat_variables_summary_csv_path = PAIRING_ROOT / "E6_alkafri_mat_variables_summary.csv"
mat_variables_summary_df.to_csv(mat_variables_summary_csv_path, index=False)
display(mat_variables_summary_df.head())

Leyendo MAT: 0it [00:00, ?it/s]

""


## Lectura de PNG ground truth

In [24]:
from PIL import Image
import numpy as np
import pandas as pd
import json
from tqdm.auto import tqdm

# ============================================================
# Lectura optimizada de PNG ground truth
# No abrir los 29k PNG completos desde Google Drive.
# Se inventarían todos por path, pero solo se leen muestras.
# ============================================================

PNG_HEADER_SAMPLE_N = 1200
PNG_PIXEL_SAMPLE_N = 250
RANDOM_SEED = 42

png_gt_df = gt_real_df[
    gt_real_df["extension"].isin([".png", ".jpg", ".jpeg", ".bmp"])
].copy() if len(gt_real_df) else pd.DataFrame()

print("PNG/JPG/BMP ground truth detectados:", len(png_gt_df))

# Excluir cosas auxiliares por seguridad
if len(png_gt_df):
    rel_lower = png_gt_df["relative_path"].astype(str).str.lower()
    name_lower = png_gt_df["file_name"].astype(str).str.lower()

    png_gt_df = png_gt_df[
        ~rel_lower.str.contains("source_code", na=False)
        & ~name_lower.str.startswith("figure")
    ].copy()

print("PNG/JPG/BMP ground truth luego de filtros:", len(png_gt_df))

# Priorizar ground truth final si existe
rel_lower = png_gt_df["relative_path"].astype(str).str.lower() if len(png_gt_df) else pd.Series(dtype=str)

final_gt_mask = (
    rel_lower.str.contains("05_final_ground_truth_data", na=False)
    | rel_lower.str.contains("ground_truth_label", na=False)
)

manual_gt_mask = rel_lower.str.contains("03_manual_label_data", na=False)

final_png_df = png_gt_df[final_gt_mask].copy()
manual_png_df = png_gt_df[manual_gt_mask].copy()

print("PNG en final/ground truth:", len(final_png_df))
print("PNG en manual label:", len(manual_png_df))

# Para muestreo de headers, priorizamos final_gt si hay suficientes.
if len(final_png_df) > 0:
    header_pool_df = final_png_df.copy()
    print("Pool elegido para headers: final/ground truth")
else:
    header_pool_df = png_gt_df.copy()
    print("Pool elegido para headers: todos los PNG filtrados")

header_sample_n = min(PNG_HEADER_SAMPLE_N, len(header_pool_df))

if header_sample_n > 0:
    header_sample_df = header_pool_df.sample(
        n=header_sample_n,
        random_state=RANDOM_SEED
    ).copy()
else:
    header_sample_df = header_pool_df.copy()

print("PNG a abrir para headers:", len(header_sample_df))

# ------------------------------------------------------------
# Crear resumen de TODOS sin abrirlos
# ------------------------------------------------------------
png_ground_truth_summary_df = png_gt_df.copy()

for col, default in {
    "shape": None,
    "mode": None,
    "width": None,
    "height": None,
    "bands": None,
    "header_read": False,
    "unique_count_approx": None,
    "unique_values_preview": None,
    "appears_binary": False,
    "appears_multiclass": False,
    "pixel_stats_computed": False,
    "error": None,
    "pixel_stats_error": None,
}.items():
    png_ground_truth_summary_df[col] = default

# ------------------------------------------------------------
# Leer headers SOLO de la muestra
# ------------------------------------------------------------
header_rows = []

for _, row in tqdm(
    header_sample_df.iterrows(),
    total=len(header_sample_df),
    desc="Leyendo headers PNG GT - muestra"
):
    item = {
        "file_path": row["file_path"],
        "shape": None,
        "mode": None,
        "width": None,
        "height": None,
        "bands": None,
        "header_read": True,
        "error": None,
    }

    try:
        with Image.open(row["file_path"]) as img:
            width, height = img.size
            mode = img.mode
            bands = len(img.getbands())

            if bands == 1:
                shape = [height, width]
            else:
                shape = [height, width, bands]

            item.update({
                "shape": json.dumps(shape),
                "mode": mode,
                "width": width,
                "height": height,
                "bands": bands,
            })

    except Exception as exc:
        item["error"] = repr(exc)

    header_rows.append(item)

header_stats_df = pd.DataFrame(header_rows)

if len(header_stats_df) > 0:
    update_cols = ["shape", "mode", "width", "height", "bands", "header_read", "error"]

    png_ground_truth_summary_df = png_ground_truth_summary_df.merge(
        header_stats_df[["file_path"] + update_cols],
        on="file_path",
        how="left",
        suffixes=("", "_sample")
    )

    for col in update_cols:
        sample_col = f"{col}_sample"
        if sample_col in png_ground_truth_summary_df.columns:
            png_ground_truth_summary_df[col] = png_ground_truth_summary_df[sample_col].combine_first(
                png_ground_truth_summary_df[col]
            )
            png_ground_truth_summary_df.drop(columns=[sample_col], inplace=True)

# ------------------------------------------------------------
# Analizar píxeles SOLO de una submuestra con header leído
# ------------------------------------------------------------
valid_header_df = png_ground_truth_summary_df[
    png_ground_truth_summary_df["header_read"] == True
].copy()

pixel_sample_n = min(PNG_PIXEL_SAMPLE_N, len(valid_header_df))

if pixel_sample_n > 0:
    pixel_sample_df = valid_header_df.sample(
        n=pixel_sample_n,
        random_state=RANDOM_SEED
    ).copy()
else:
    pixel_sample_df = valid_header_df.copy()

print("PNG a abrir para análisis de píxeles:", len(pixel_sample_df))

pixel_rows = []

for _, row in tqdm(
    pixel_sample_df.iterrows(),
    total=len(pixel_sample_df),
    desc="Analizando píxeles PNG GT - muestra"
):
    pixel_item = {
        "file_path": row["file_path"],
        "unique_count_approx": None,
        "unique_values_preview": None,
        "appears_binary": False,
        "appears_multiclass": False,
        "pixel_stats_computed": True,
        "pixel_stats_error": None,
    }

    try:
        with Image.open(row["file_path"]) as img:
            arr = np.asarray(img)

        if arr.ndim == 2:
            unique_values = np.unique(arr)
            unique_count = int(len(unique_values))
            preview = unique_values[:30].tolist()

            values_set = set(int(v) for v in unique_values[:100])
            appears_binary = unique_count <= 3 and values_set.issubset({0, 1, 255})
            appears_multiclass = unique_count > 3 and unique_count <= 30

            pixel_item.update({
                "unique_count_approx": unique_count,
                "unique_values_preview": json.dumps(preview),
                "appears_binary": bool(appears_binary),
                "appears_multiclass": bool(appears_multiclass),
            })

        else:
            flat = arr.reshape(-1, arr.shape[-1])

            max_pixels = 50_000
            if flat.shape[0] > max_pixels:
                rng = np.random.default_rng(RANDOM_SEED)
                idx = rng.choice(flat.shape[0], size=max_pixels, replace=False)
                flat = flat[idx]

            unique_colors = np.unique(flat, axis=0)
            unique_count = int(len(unique_colors))
            preview = unique_colors[:20].tolist()

            appears_binary = unique_count <= 3
            appears_multiclass = unique_count > 3 and unique_count <= 50

            pixel_item.update({
                "unique_count_approx": unique_count,
                "unique_values_preview": json.dumps(preview),
                "appears_binary": bool(appears_binary),
                "appears_multiclass": bool(appears_multiclass),
            })

    except Exception as exc:
        pixel_item["pixel_stats_error"] = repr(exc)

    pixel_rows.append(pixel_item)

pixel_stats_df = pd.DataFrame(pixel_rows)

if len(pixel_stats_df) > 0:
    cols_to_update = [
        "unique_count_approx",
        "unique_values_preview",
        "appears_binary",
        "appears_multiclass",
        "pixel_stats_computed",
        "pixel_stats_error",
    ]

    png_ground_truth_summary_df = png_ground_truth_summary_df.merge(
        pixel_stats_df[["file_path"] + cols_to_update],
        on="file_path",
        how="left",
        suffixes=("", "_sample")
    )

    for col in cols_to_update:
        sample_col = f"{col}_sample"
        if sample_col in png_ground_truth_summary_df.columns:
            png_ground_truth_summary_df[col] = png_ground_truth_summary_df[sample_col].combine_first(
                png_ground_truth_summary_df[col]
            )
            png_ground_truth_summary_df.drop(columns=[sample_col], inplace=True)

png_ground_truth_summary_df["header_read"] = png_ground_truth_summary_df["header_read"].fillna(False)
png_ground_truth_summary_df["pixel_stats_computed"] = png_ground_truth_summary_df["pixel_stats_computed"].fillna(False)

png_ground_truth_summary_csv_path = PAIRING_ROOT / "E6_alkafri_png_ground_truth_summary.csv"
png_ground_truth_summary_df.to_csv(png_ground_truth_summary_csv_path, index=False)

print("PNG ground truth summary:", png_ground_truth_summary_csv_path)
print("Total PNG/JPG/BMP resumidos:", len(png_ground_truth_summary_df))
print("Headers leídos:", int(png_ground_truth_summary_df["header_read"].sum()))
print("Con análisis de píxeles:", int(png_ground_truth_summary_df["pixel_stats_computed"].sum()))

display(png_ground_truth_summary_df.head())

PNG/JPG/BMP ground truth detectados: 29355
PNG/JPG/BMP ground truth luego de filtros: 29355
PNG en final/ground truth: 13905
PNG en manual label: 15450
Pool elegido para headers: final/ground truth
PNG a abrir para headers: 1200


Leyendo headers PNG GT - muestra:   0%|          | 0/1200 [00:00<?, ?it/s]

PNG a abrir para análisis de píxeles: 250


Analizando píxeles PNG GT - muestra:   0%|          | 0/250 [00:00<?, ?it/s]

/tmp/ipykernel_8201/3277177414.py:269: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  png_ground_truth_summary_df["header_read"] = png_ground_truth_summary_df["header_read"].fillna(False)
/tmp/ipykernel_8201/3277177414.py:270: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  png_ground_truth_summary_df["pixel_stats_computed"] = png_ground_truth_summary_df["pixel_stats_computed"].fillna(False)


PNG ground truth summary: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_png_ground_truth_summary.csv
Total PNG/JPG/BMP resumidos: 29355
Headers leídos: 1200
Con análisis de píxeles: 250


,file_path,relative_path,file_name,extension,parent_folder,size_bytes,probable_source,probable_type,is_nested,nested_source_zip,has_axial,has_sagittal,has_sag,has_mask,has_label,has_ground,has_gt,has_segmentation,has_disc,has_posterior,has_thecal,has_canal,has_intervertebral,has_l3,has_l4,has_l5,has_s1,shape,mode,width,height,bands,header_read,unique_count_approx,unique_values_preview,appears_binary,appears_multiclass,pixel_stats_computed,error,pixel_stats_error
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D3.png,.png,Labeller 01,1019,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,None,None,None,None,None,False,None,None,False,False,False,None,None
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D4.png,.png,Labeller 01,1115,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,None,None,None,None,None,False,None,None,False,False,False,None,None
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0001_D5.png,.png,Labeller 01,1084,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,None,None,None,None,None,False,None,None,False,False,False,None,None
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0002_D3.png,.png,Labeller 01,1044,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,None,None,None,None,None,False,None,None,False,False,False,None,None
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,T1_0002_D4.png,.png,Labeller 01,1065,ground_truth,mask_or_label_image,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,None,None,None,None,None,False,None,None,False,False,False,None,None


## Tokens de imagenes y ground truth

In [25]:
def extract_path_tokens(row):
    text = f"{row.get('relative_path', '')} {row.get('file_path', '')} {row.get('file_name', '')}"
    tokens = numeric_tokens_from_text(text)
    patient = row.get("PatientID") if "PatientID" in row and pd.notna(row.get("PatientID")) else (tokens[0] if tokens else None)
    instance = row.get("InstanceNumber") if "InstanceNumber" in row and pd.notna(row.get("InstanceNumber")) else (tokens[-1] if tokens else None)
    return {
        "file_path": row.get("file_path"),
        "relative_path": row.get("relative_path"),
        "patient_id_candidate": str(patient) if patient is not None else None,
        "series_id_candidate": row.get("SeriesInstanceUID") if "SeriesInstanceUID" in row else None,
        "slice_id_candidate": str(instance) if instance is not None else None,
        "numeric_tokens": json.dumps(tokens),
        "level_tokens": json.dumps([t.upper() for t in tokens if re.match(r"l[1-5]|s1", t, re.I)]),
    }


image_path_tokens_df = pd.DataFrame([extract_path_tokens(row) for _, row in axial_ima_df.iterrows()])
gt_path_tokens_pairing_df = pd.DataFrame([extract_path_tokens(row) for _, row in gt_real_df.iterrows()])

image_path_tokens_csv_path = PAIRING_ROOT / "E6_alkafri_image_path_tokens.csv"
gt_path_tokens_pairing_csv_path = PAIRING_ROOT / "E6_alkafri_gt_path_tokens.csv"
image_path_tokens_df.to_csv(image_path_tokens_csv_path, index=False)
gt_path_tokens_pairing_df.to_csv(gt_path_tokens_pairing_csv_path, index=False)
display(image_path_tokens_df.head())
display(gt_path_tokens_pairing_df.head())

,file_path,relative_path,patient_id_candidate,series_id_candidate,slice_id_candidate,numeric_tokens,level_tokens
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,01,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,1,"[""01"", ""0037"", ""20150919"", ""125616"", ""225000"",...",[]
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,01,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,2,"[""01"", ""0037"", ""20150919"", ""125616"", ""225000"",...",[]
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,01,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,3,"[""01"", ""0037"", ""20150919"", ""125616"", ""225000"",...",[]
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,01,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,4,"[""01"", ""0037"", ""20150919"", ""125616"", ""225000"",...",[]
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/003...,01,1.3.12.2.1107.5.2.40.50233.2015091913064063223...,5,"[""01"", ""0037"", ""20150919"", ""125616"", ""225000"",...",[]


,file_path,relative_path,patient_id_candidate,series_id_candidate,slice_id_candidate,numeric_tokens,level_tokens
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,03,None,3,"[""03"", ""01"", ""1"", ""0001"", ""3"", ""03"", ""01"", ""1""...",[]
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,03,None,3,"[""03"", ""01"", ""1"", ""0001"", ""3"", ""03"", ""01"", ""1""...",[]
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,03,None,4,"[""03"", ""01"", ""1"", ""0001"", ""4"", ""03"", ""01"", ""1""...",[]
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,03,None,4,"[""03"", ""01"", ""1"", ""0001"", ""4"", ""03"", ""01"", ""1""...",[]
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Manual_Label_Data/03_Man...,03,None,5,"[""03"", ""01"", ""1"", ""0001"", ""5"", ""03"", ""01"", ""1""...",[]


## Estrategias de emparejamiento candidatas

In [27]:
import json
import re
import pandas as pd
from tqdm.auto import tqdm

# ============================================================
# Estrategias de pairing optimizadas
# Evita loops imagen x ground truth completos.
# ============================================================

MAX_ROWS_PER_STRATEGY = 500_000
MAX_TOKEN_FREQ_IMAGES = 500
MAX_TOKEN_FREQ_GT = 500

def safe_json_list(value):
    try:
        if pd.isna(value):
            return []
        parsed = json.loads(value) if isinstance(value, str) else value
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
        return []
    except Exception:
        return []

def normalize_token(token):
    token = str(token)
    token = token.strip()
    if token == "":
        return None
    # Normaliza ceros a la izquierda: 0001 -> 1
    normalized = token.lstrip("0")
    return normalized if normalized else "0"

def parse_shape(value):
    if pd.isna(value):
        return None
    nums = [int(x) for x in re.findall(r"\d+", str(value))]
    return tuple(nums) if nums else None

def image_shape_from_row(row):
    try:
        rows = row.get("Rows")
        cols = row.get("Columns")
        if pd.isna(rows) or pd.isna(cols):
            return None
        return (int(float(rows)), int(float(cols)))
    except Exception:
        return None

# ------------------------------------------------------------
# Ground truth shape lookup optimizado
# ------------------------------------------------------------

gt_shape_rows = []

if "png_ground_truth_summary_df" in globals() and len(png_ground_truth_summary_df):
    png_shape_df = png_ground_truth_summary_df.copy()
    if "shape" in png_shape_df.columns:
        for _, row in png_shape_df[png_shape_df["shape"].notna()].iterrows():
            shape = parse_shape(row.get("shape"))
            if shape:
                gt_shape_rows.append({
                    "gt_file_path": row["file_path"],
                    "mask_shape": tuple(shape[:2]),
                    "gt_source": "png"
                })

if "mat_variables_summary_df" in globals() and len(mat_variables_summary_df):
    mat_mask_df = mat_variables_summary_df[
        mat_variables_summary_df["mask_candidate"].fillna(False)
    ].copy()
    for _, row in mat_mask_df.iterrows():
        shape = parse_shape(row.get("shape"))
        if shape:
            gt_shape_rows.append({
                "gt_file_path": row["file_path"],
                "mask_shape": tuple(shape[:2]),
                "gt_source": "mat"
            })

gt_shape_df = pd.DataFrame(gt_shape_rows).drop_duplicates() if gt_shape_rows else pd.DataFrame(
    columns=["gt_file_path", "mask_shape", "gt_source"]
)

print("GT con shape disponible:", len(gt_shape_df))

# ------------------------------------------------------------
# Estrategia A: PatientID / carpeta
# Usar merge, no loop anidado.
# ------------------------------------------------------------

img_patient_df = image_path_tokens_df.copy()
gt_patient_df = gt_path_tokens_pairing_df.copy()

img_patient_df["patient_id_candidate"] = img_patient_df["patient_id_candidate"].astype(str)
gt_patient_df["patient_id_candidate"] = gt_patient_df["patient_id_candidate"].astype(str)

img_patient_df = img_patient_df[
    img_patient_df["patient_id_candidate"].notna()
    & ~img_patient_df["patient_id_candidate"].isin(["", "nan", "None"])
].copy()

gt_patient_df = gt_patient_df[
    gt_patient_df["patient_id_candidate"].notna()
    & ~gt_patient_df["patient_id_candidate"].isin(["", "nan", "None"])
].copy()

strategy_a_df = img_patient_df[["file_path", "patient_id_candidate"]].rename(
    columns={"file_path": "image_file_path"}
).merge(
    gt_patient_df[["file_path", "patient_id_candidate"]].rename(
        columns={"file_path": "gt_file_path"}
    ),
    on="patient_id_candidate",
    how="inner"
)

strategy_a_df["reason"] = "patient_or_folder_token_match"

if len(strategy_a_df) > MAX_ROWS_PER_STRATEGY:
    strategy_a_df = strategy_a_df.head(MAX_ROWS_PER_STRATEGY).copy()

strategy_a_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_a_patient.csv"
strategy_a_df.to_csv(strategy_a_csv_path, index=False)

print("Strategy A rows:", len(strategy_a_df))

# ------------------------------------------------------------
# Estrategia B: tokens numéricos compartidos optimizado
# Explode + merge + filtro de tokens demasiado frecuentes.
# ------------------------------------------------------------

img_tokens_df = image_path_tokens_df[["file_path", "numeric_tokens"]].copy()
gt_tokens_df = gt_path_tokens_pairing_df[["file_path", "numeric_tokens"]].copy()

img_tokens_df["token"] = img_tokens_df["numeric_tokens"].apply(safe_json_list)
gt_tokens_df["token"] = gt_tokens_df["numeric_tokens"].apply(safe_json_list)

img_tokens_df = img_tokens_df.explode("token")
gt_tokens_df = gt_tokens_df.explode("token")

img_tokens_df["token"] = img_tokens_df["token"].apply(normalize_token)
gt_tokens_df["token"] = gt_tokens_df["token"].apply(normalize_token)

img_tokens_df = img_tokens_df.dropna(subset=["token"])
gt_tokens_df = gt_tokens_df.dropna(subset=["token"])

# Excluir tokens poco informativos y extremadamente frecuentes
bad_tokens = set(["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"])
bad_tokens |= set(["320", "384", "512", "256", "224"])

img_tokens_df = img_tokens_df[~img_tokens_df["token"].isin(bad_tokens)].copy()
gt_tokens_df = gt_tokens_df[~gt_tokens_df["token"].isin(bad_tokens)].copy()

img_freq = img_tokens_df["token"].value_counts()
gt_freq = gt_tokens_df["token"].value_counts()

valid_tokens = set(img_freq[img_freq <= MAX_TOKEN_FREQ_IMAGES].index) & set(
    gt_freq[gt_freq <= MAX_TOKEN_FREQ_GT].index
)

img_tokens_df = img_tokens_df[img_tokens_df["token"].isin(valid_tokens)].copy()
gt_tokens_df = gt_tokens_df[gt_tokens_df["token"].isin(valid_tokens)].copy()

img_tokens_df = img_tokens_df.rename(columns={"file_path": "image_file_path"})
gt_tokens_df = gt_tokens_df.rename(columns={"file_path": "gt_file_path"})

strategy_b_df = img_tokens_df[["image_file_path", "token"]].merge(
    gt_tokens_df[["gt_file_path", "token"]],
    on="token",
    how="inner"
)

strategy_b_df = (
    strategy_b_df
    .groupby(["image_file_path", "gt_file_path"], as_index=False)
    .agg(shared_tokens=("token", lambda x: json.dumps(sorted(set(map(str, x))))))
)

strategy_b_df["reason"] = "numeric_token_overlap_filtered"

if len(strategy_b_df) > MAX_ROWS_PER_STRATEGY:
    strategy_b_df = strategy_b_df.head(MAX_ROWS_PER_STRATEGY).copy()

strategy_b_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_b_tokens.csv"
strategy_b_df.to_csv(strategy_b_csv_path, index=False)

print("Strategy B rows:", len(strategy_b_df))
print("Tokens válidos usados:", len(valid_tokens))

# ------------------------------------------------------------
# Estrategia C: shape match optimizada
# Crear shape de imagen y merge por shape.
# ------------------------------------------------------------

img_shape_rows = []
for _, img in axial_ima_df.iterrows():
    img_shape = image_shape_from_row(img)
    if img_shape is not None:
        img_shape_rows.append({
            "image_file_path": img["file_path"],
            "image_shape": tuple(img_shape),
        })

img_shape_df = pd.DataFrame(img_shape_rows).drop_duplicates() if img_shape_rows else pd.DataFrame(
    columns=["image_file_path", "image_shape"]
)

strategy_c_df = img_shape_df.merge(
    gt_shape_df,
    left_on="image_shape",
    right_on="mask_shape",
    how="inner"
)

strategy_c_df["reason"] = "shape_match"

if len(strategy_c_df) > MAX_ROWS_PER_STRATEGY:
    strategy_c_df = strategy_c_df.head(MAX_ROWS_PER_STRATEGY).copy()

strategy_c_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_c_shape.csv"
strategy_c_df.to_csv(strategy_c_csv_path, index=False)

print("Strategy C rows:", len(strategy_c_df))

# ------------------------------------------------------------
# Estrategia D: metadata source code
# Esta parte es chica y puede quedar igual.
# ------------------------------------------------------------

metadata_files = extracted_inventory_df[
    extracted_inventory_df["file_name"].astype(str).str.lower().isin([
        "slices numbers.csv",
        "t1_subfolders.csv",
        "t2_subfolders.csv",
        "png counts.csv"
    ])
].copy()

strategy_d_rows = []

for _, row in metadata_files.iterrows():
    try:
        preview = pd.read_csv(row["file_path"], nrows=10).to_json(
            orient="records",
            force_ascii=False
        )
    except Exception as exc:
        preview = repr(exc)

    strategy_d_rows.append({
        "metadata_file_path": row["file_path"],
        "relative_path": row["relative_path"],
        "file_name": row["file_name"],
        "preview": preview,
        "reason": "source_metadata_hint",
    })

strategy_d_df = pd.DataFrame(strategy_d_rows)

strategy_d_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_d_source_metadata.csv"
strategy_d_df.to_csv(strategy_d_csv_path, index=False)

print("Strategy D rows:", len(strategy_d_df))

print("\nCSVs exportados:")
print("-", strategy_a_csv_path)
print("-", strategy_b_csv_path)
print("-", strategy_c_csv_path)
print("-", strategy_d_csv_path)

GT con shape disponible: 1200
Strategy A rows: 0
Strategy B rows: 500000
Tokens válidos usados: 166
Strategy C rows: 500000
Strategy D rows: 4

CSVs exportados:
- /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_pairing_strategy_a_patient.csv
- /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_pairing_strategy_b_tokens.csv
- /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_pairing_strategy_c_shape.csv
- /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_pairing_strategy_d_source_metadata.csv


## Seleccion de pairing preliminar

In [28]:
signals = {}

def add_signal(image_path, gt_path, signal_name):
    key = (image_path, gt_path)
    signals.setdefault(key, set()).add(signal_name)

for _, row in strategy_a_df.iterrows():
    add_signal(row["image_file_path"], row["gt_file_path"], "patient")
for _, row in strategy_b_df.iterrows():
    add_signal(row["image_file_path"], row["gt_file_path"], "tokens")
for _, row in strategy_c_df.iterrows():
    add_signal(row["image_file_path"], row["gt_file_path"], "shape")

img_lookup = axial_ima_df.set_index("file_path").to_dict(orient="index") if len(axial_ima_df) else {}
gt_lookup = gt_real_df.set_index("file_path").to_dict(orient="index") if len(gt_real_df) else {}

pair_rows = []
for (img_path, gt_path), sigs in signals.items():
    img = img_lookup.get(img_path, {})
    gt = gt_lookup.get(gt_path, {})
    image_shape = image_shape_from_row(img)
    mask_shape = gt_shape_lookup.get(gt_path)
    shape_match = bool(image_shape is not None and mask_shape is not None and tuple(image_shape) == tuple(mask_shape))
    n_signals = len(sigs)
    if {"patient", "shape", "tokens"}.issubset(sigs):
        confidence = "alta"
    elif n_signals >= 2:
        confidence = "media"
    else:
        confidence = "baja"
    needs_manual_review = confidence != "alta"
    pair_rows.append({
        "image_file_path": img_path,
        "image_relative_path": img.get("relative_path"),
        "gt_file_path": gt_path,
        "gt_relative_path": gt.get("relative_path"),
        "pairing_strategy": "|".join(sorted(sigs)),
        "patient_id_candidate": img.get("PatientID"),
        "series_id_candidate": img.get("SeriesInstanceUID"),
        "slice_id_candidate": img.get("InstanceNumber"),
        "image_shape": str(image_shape),
        "mask_shape": str(mask_shape),
        "shape_match": shape_match,
        "confidence": confidence,
        "reason": f"signals={sorted(sigs)}",
        "needs_manual_review": needs_manual_review,
    })

pairing_candidates_df = pd.DataFrame(pair_rows)
if len(pairing_candidates_df) > 0:
    pairing_candidates_df = pairing_candidates_df.sort_values(["confidence", "shape_match"], ascending=[True, False])

pairing_candidates_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_candidates.csv"
pairing_candidates_df.to_csv(pairing_candidates_csv_path, index=False)

print("Pairing candidates:", len(pairing_candidates_df))
display(pairing_candidates_df.head(20))

Pairing candidates: 999282


,image_file_path,image_relative_path,gt_file_path,gt_relative_path,pairing_strategy,patient_id_candidate,series_id_candidate,slice_id_candidate,image_shape,mask_shape,shape_match,confidence,reason,needs_manual_review
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/04_In...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016042414585564183...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
12,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/04_In...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016042414585564183...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
18,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/05_Fi...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016042414585564183...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
147,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/04_In...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016042414560235597...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
156,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/04_In...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016042414560235597...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
162,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/05_Fi...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016042414560235597...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
291,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/001...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/04_In...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016021010072583969...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
300,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/001...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/04_In...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016021010072583969...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
306,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/001...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/05_Fi...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016021010072583969...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True
435,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/001...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/ground_truth__Ground_Truth_Label/04_In...,tokens,NaN,1.3.12.2.1107.5.2.40.50233.2016021010043318596...,17,"(320, 320)","(320, 320)",True,baja,signals=['tokens'],True


## Visualizacion de pairing candidatos

In [29]:
def normalize_percentile(arr, p_low=1, p_high=99):
    arr = arr.astype(np.float32)
    low, high = np.percentile(arr, [p_low, p_high])
    if np.isclose(low, high):
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - low) / (high - low), 0, 1).astype(np.float32)


def read_image_ima(path):
    ds = pydicom.dcmread(str(path), force=True)
    return normalize_percentile(ds.pixel_array), ds


def read_mask_any(path):
    path = str(path)
    ext = Path(path).suffix.lower()
    if ext in [".png", ".jpg", ".jpeg", ".bmp"]:
        return np.asarray(Image.open(path))
    if ext == ".mat":
        data = loadmat(path)
        for key, value in data.items():
            if key.startswith("__"):
                continue
            arr = np.asarray(value)
            if np.issubdtype(arr.dtype, np.number) and arr.ndim in [2, 3]:
                return arr[:, :, 0] if arr.ndim == 3 else arr
    raise ValueError(f"No se pudo leer mascara: {path}")


def plot_pair(image, mask, title, output_path):
    mask_2d = mask[:, :, 0] if mask.ndim == 3 else mask
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Imagen axial")
    axes[1].imshow(mask_2d, cmap="viridis")
    axes[1].set_title("Ground truth")
    axes[2].imshow(image, cmap="gray")
    overlay = np.ma.masked_where(mask_2d == 0, mask_2d)
    axes[2].imshow(overlay, cmap="autumn", alpha=0.45)
    axes[2].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


pairing_figure_paths = []
visualizable_pairs = pairing_candidates_df[pairing_candidates_df["confidence"].isin(["alta", "media"])].copy() if len(pairing_candidates_df) else pd.DataFrame()
for idx, row in enumerate(visualizable_pairs.head(10).itertuples(index=False), start=1):
    output_path = FIGURES_ROOT / f"E6_alkafri_pairing_example_{idx:02d}.png"
    try:
        image, ds = read_image_ima(row.image_file_path)
        mask = read_mask_any(row.gt_file_path)
        plot_pair(image, mask, f"{row.confidence} - {row.pairing_strategy}", output_path)
        pairing_figure_paths.append(str(output_path))
    except Exception as exc:
        print("No se pudo visualizar par:", row.image_file_path, row.gt_file_path, repr(exc))

if not pairing_figure_paths:
    axial_examples_path = FIGURES_ROOT / "E6_alkafri_unpaired_axial_image_examples.png"
    gt_examples_path = FIGURES_ROOT / "E6_alkafri_unpaired_ground_truth_examples.png"
    fig, axes = plt.subplots(1, min(5, len(axial_ima_df)), figsize=(15, 3)) if len(axial_ima_df) else (None, [])
    if len(axial_ima_df):
        axes = np.atleast_1d(axes)
        for ax, (_, row) in zip(axes, axial_ima_df.head(5).iterrows()):
            try:
                image, _ = read_image_ima(row["file_path"])
                ax.imshow(image, cmap="gray")
            except Exception:
                ax.text(0.5, 0.5, "error", ha="center")
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(axial_examples_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        pairing_figure_paths.append(str(axial_examples_path))
    fig, axes = plt.subplots(1, min(5, len(gt_real_df)), figsize=(15, 3)) if len(gt_real_df) else (None, [])
    if len(gt_real_df):
        axes = np.atleast_1d(axes)
        for ax, (_, row) in zip(axes, gt_real_df.head(5).iterrows()):
            try:
                mask = read_mask_any(row["file_path"])
                ax.imshow(mask[:, :, 0] if mask.ndim == 3 else mask, cmap="viridis")
            except Exception:
                ax.text(0.5, 0.5, "error", ha="center")
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(gt_examples_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        pairing_figure_paths.append(str(gt_examples_path))

print("Figuras:", pairing_figure_paths)

Figuras: ['/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_01.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_02.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_03.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_04.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_05.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_06.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_07.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_08.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_09.png', '/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_pairing_example_10.png']


## Dataset axial procesado preliminar

In [30]:
processed_rows = []
trusted_pairs = pairing_candidates_df[pairing_candidates_df["confidence"].isin(["alta", "media"])].copy() if len(pairing_candidates_df) else pd.DataFrame()
pairing_v1_created = False

if len(trusted_pairs) > 0:
    PAIRING_V1_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, row in enumerate(trusted_pairs.itertuples(index=False), start=1):
        sample_id = f"pair_{idx:04d}"
        try:
            image, ds = read_image_ima(row.image_file_path)
            mask = read_mask_any(row.gt_file_path)
            image_out = PAIRING_V1_ROOT / f"{sample_id}_image.npy"
            mask_out = PAIRING_V1_ROOT / f"{sample_id}_mask.npy"
            np.save(image_out, image.astype(np.float32))
            np.save(mask_out, mask)
            processed_rows.append({
                "sample_id": sample_id,
                "image_npy": str(image_out),
                "mask_npy": str(mask_out),
                "image_file_path": row.image_file_path,
                "gt_file_path": row.gt_file_path,
                "confidence": row.confidence,
                "pairing_strategy": row.pairing_strategy,
                "image_shape": str(image.shape),
                "mask_shape": str(mask.shape),
            })
        except Exception as exc:
            processed_rows.append({
                "sample_id": sample_id,
                "image_file_path": row.image_file_path,
                "gt_file_path": row.gt_file_path,
                "confidence": row.confidence,
                "pairing_strategy": row.pairing_strategy,
                "error": repr(exc),
            })
    pairing_v1_created = any("image_npy" in row for row in processed_rows)
else:
    print("No hay pares alta/media; no se crea dataset final pairing_v1.")

processed_pairing_index_df = pd.DataFrame(processed_rows)
processed_pairing_index_csv_path = PAIRING_ROOT / "E6_alkafri_processed_pairing_v1_index.csv"
processed_pairing_index_df.to_csv(processed_pairing_index_csv_path, index=False)
display(processed_pairing_index_df.head())

,sample_id,image_npy,mask_npy,image_file_path,gt_file_path,confidence,pairing_strategy,image_shape,mask_shape
0,pair_0001,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,media,shape|tokens,"(320, 320)","(320, 320)"
1,pair_0002,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,media,shape|tokens,"(320, 320)","(320, 320)"
2,pair_0003,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,media,shape|tokens,"(320, 320)","(320, 320)"
3,pair_0004,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,media,shape|tokens,"(320, 320)","(320, 320)"
4,pair_0005,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,media,shape|tokens,"(320, 320)","(320, 320)"


## Sanity checks, reporte y conclusion

In [31]:
confidence_counts = pairing_candidates_df["confidence"].value_counts().to_dict() if len(pairing_candidates_df) else {}
problems = []
if len(pairing_candidates_df) == 0:
    problems.append("No se encontraron pares candidatos.")
if len(pairing_candidates_df) and not pairing_candidates_df["shape_match"].any():
    problems.append("No hay pares con shape match.")
if len(strategy_d_df) > 0:
    problems.append("Se requiere interpretar metadata/codigo MATLAB para confirmar pairing.")
if len(gt_real_df) > 0 and len(gt_shape_lookup) == 0:
    problems.append("Ground truth en formato inesperado o sin shape legible.")

sanity_checks = {
    "n_axial_series": int(len(axial_series_summary_df)),
    "n_axial_ima_candidates": int(len(axial_ima_df)),
    "n_ground_truth_real": int(len(gt_real_df)),
    "n_png_ground_truth": int(len(png_ground_truth_summary_df)),
    "n_mat_ground_truth": int(len(mat_files)),
    "n_pairing_candidates": int(len(pairing_candidates_df)),
    "n_pairs_alta": int(confidence_counts.get("alta", 0)),
    "n_pairs_media": int(confidence_counts.get("media", 0)),
    "n_pairs_baja": int(confidence_counts.get("baja", 0)),
    "n_shape_match": int(pairing_candidates_df["shape_match"].sum()) if len(pairing_candidates_df) else 0,
    "n_pairs_visualized": int(len(pairing_figure_paths)),
    "pairing_v1_created": bool(pairing_v1_created),
    "problems": problems,
}
sanity_checks_json_path = PAIRING_ROOT / "E6_alkafri_pairing_sanity_checks.json"
with open(sanity_checks_json_path, "w", encoding="utf-8") as f:
    json.dump(sanity_checks, f, indent=2, ensure_ascii=False)

exports = {
    "axial_ima_candidates": str(axial_ima_candidates_csv_path),
    "axial_series_summary": str(axial_series_summary_csv_path),
    "ground_truth_real_files": str(gt_real_csv_path),
    "ground_truth_path_tokens": str(gt_path_tokens_csv_path),
    "ground_truth_folder_summary": str(gt_folder_summary_csv_path),
    "mat_variables_summary": str(mat_variables_summary_csv_path),
    "png_ground_truth_summary": str(png_ground_truth_summary_csv_path),
    "image_path_tokens": str(image_path_tokens_csv_path),
    "gt_path_tokens": str(gt_path_tokens_pairing_csv_path),
    "strategy_a": str(strategy_a_csv_path),
    "strategy_b": str(strategy_b_csv_path),
    "strategy_c": str(strategy_c_csv_path),
    "strategy_d": str(strategy_d_csv_path),
    "pairing_candidates": str(pairing_candidates_csv_path),
    "processed_pairing_v1_index": str(processed_pairing_index_csv_path),
    "sanity_checks": str(sanity_checks_json_path),
    "figures": pairing_figure_paths,
}

recommendation_14 = (
    "Entrenar notebook 14 solo con pairing_v1 alta/media y revision manual previa."
    if pairing_v1_created else
    "No entrenar todavia; resolver ambiguedad de pairing interpretando metadata y codigo MATLAB."
)

report = {
    "inputs": {k: str(v) for k, v in INPUTS.items()},
    "n_valid_dicom_ima": int(summary["valid_dicom_orientation_rows"]),
    "n_axial_candidates": int(len(axial_ima_df)),
    "n_axial_series": int(len(axial_series_summary_df)),
    "n_ground_truth": int(len(gt_real_df)),
    "strategy_results": {
        "patient": int(len(strategy_a_df)),
        "tokens": int(len(strategy_b_df)),
        "shape": int(len(strategy_c_df)),
        "source_metadata": int(len(strategy_d_df)),
    },
    "confidence_counts": confidence_counts,
    "processed_dataset_created": bool(pairing_v1_created),
    "exports": exports,
    "recommendation_for_notebook_14": recommendation_14,
}
report_json_path = PAIRING_ROOT / "E6_alkafri_pairing_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Sanity:", sanity_checks_json_path)
print("Report:", report_json_path)

Sanity: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_pairing_sanity_checks.json
Report: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing/E6_alkafri_pairing_report.json


In [32]:
series_table_md = axial_series_summary_df.head(20).to_markdown(index=False) if len(axial_series_summary_df) else "Sin series axiales."
gt_table_md = gt_folder_summary_df.to_markdown(index=False) if len(gt_folder_summary_df) else "Sin ground truth real."
pairing_table_md = pairing_candidates_df.head(20).to_markdown(index=False) if len(pairing_candidates_df) else "Sin pares candidatos confiables."

conclusion_text = f'''# Conclusion tecnica - E6 Pairing axial Al-Kafri/Sudirman

## Objetivo

Construir el primer preprocesamiento axial trazable de Al-Kafri/Sudirman a partir de los inventarios del notebook 12, resolviendo candidatos de emparejamiento entre imagenes `.ima` axiales y ground truth real.

## Por que se hace despues del inventario

El notebook 12 confirmo que hay DICOM/IMA axiales validos y ground truth extraido desde ZIPs internos, pero el emparejamiento imagen-mascara no estaba resuelto. Este notebook separa exploracion de inventario y decision de pairing para evitar asumir relaciones sin evidencia.

## Metodologia de pairing

Se evaluaron estrategias por ID/carpeta de paciente, tokens numericos de path, match de dimensiones y metadata derivada de archivos auxiliares del codigo fuente. La confianza alta/media/baja se asigno segun cantidad y calidad de senales coincidentes.

## Hallazgos de imagenes axiales

{series_table_md}

## Hallazgos de ground truth

{gt_table_md}

## Resultado de pairing

{pairing_table_md}

Dataset `pairing_v1` creado: {pairing_v1_created}.

## Ejemplos visuales

{pairing_figure_paths}

## Limitaciones

- No se entrenan modelos.
- No se fuerza pairing definitivo si hay ambiguedad.
- Algunas mascaras pueden requerir interpretar `.mat` o scripts MATLAB.
- Shape match no es suficiente por si solo.
- Se requiere revision manual/profesional antes de usar pares para entrenamiento.

## Recomendacion

{recommendation_14}
'''

conclusion_path = DOCS_ROOT / "E6_alkafri_pairing_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)

print("Conclusion:", conclusion_path)

Conclusion: /content/drive/MyDrive/PFI_MVP/docs/E6_alkafri_pairing_conclusion.md


## Criterio de aceptacion

Este notebook cumple si carga los inventarios del notebook 12, filtra `.ima` axiales reales, excluye localizers/source_code, inspecciona PNG/MAT de ground truth real, propone estrategias trazables de pairing, exporta candidatos con confianza, genera visualizaciones y no entrena modelos.